# 🤖 Notebook 02: Tiền xử lý Underthesea & Gán nhãn cảm xúc với PhoBERT
**Dự án:** Phân tích quan điểm sinh viên CMC (`cmc-sentiment-analysis`)  
**Đầu vào:** `data/cmc_dataset_clean.csv` (1,083 bài viết và bình luận đã làm sạch)  
**Đầu ra:** `data/cmc_dataset_labeled.csv` (Bổ sung tách từ tiếng Việt, phân loại chủ đề và gán nhãn cảm xúc `Tích cực`, `Bình thường`, `Tiêu cực`)

---

## 🧩 Cell 1: Đọc dữ liệu sạch `data/cmc_dataset_clean.csv` và khởi tạo bộ phân tích PhoBERT + Underthesea

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from model.sentiment import PhoBERTSentimentAnalyzer
from data.label_dataset import classify_topic

CLEAN_CSV_PATH = BASE_DIR / "data" / "cmc_dataset_clean.csv"
LABELED_CSV_PATH = BASE_DIR / "data" / "cmc_dataset_labeled.csv"

df = pd.read_csv(CLEAN_CSV_PATH, encoding="utf-8-sig")
print(f"Đã đọc {len(df):,} bản ghi từ {CLEAN_CSV_PATH.name}")
df.head(5)

## 🏷️ Cell 2: Phân loại chủ đề (Topic) và Suy luận nhãn cảm xúc (`Tích cực`, `Bình thường`, `Tiêu cực`)
- Đặt `use_pretrained_backbone=True` nếu muốn tải trực tiếp trọng số `vinai/phobert-base-v2` từ HuggingFace, hoặc `False` để chạy chế độ Hybrid Lexicon tốc độ cao.

In [ ]:
analyzer = PhoBERTSentimentAnalyzer(
    model_name_or_path="vinai/phobert-base-v2",
    use_pretrained_backbone=False
)

results = [analyzer.predict(text) for text in df["content"].fillna("")]

df["topic"] = df["content"].fillna("").apply(classify_topic)
df["cleaned_content"] = [r.preprocessed_text for r in results]
df["label"] = [r.label for r in results]
df["sentiment"] = df["label"]
df["confidence"] = [r.confidence for r in results]

print("Phân bố nhãn cảm xúc:")
print(df["label"].value_counts())
df[["id", "type", "topic", "content", "cleaned_content", "label", "confidence"]].head(10)

## 📊 Cell 3: Trực quan hóa phân bố cảm xúc và chủ đề thảo luận của sinh viên CMC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# 1. Biểu đồ tròn tỷ lệ cảm xúc
sentiment_counts = df["label"].value_counts()
color_map = {"Tích cực": "#10B981", "Bình thường": "#3B82F6", "Tiêu cực": "#EF4444"}
pie_colors = [color_map.get(lbl, "#9CA3AF") for lbl in sentiment_counts.index]

axes[0].pie(
    sentiment_counts.values,
    labels=sentiment_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=pie_colors,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
axes[0].set_title("Tỷ lệ Cảm xúc Sinh viên CMC (1,083 bản ghi)", fontsize=12, fontweight="bold")

# 2. Biểu đồ cột ngang theo chủ đề
topic_sentiment = pd.crosstab(df["topic"], df["label"])
ordered_cols = [c for c in ["Tích cực", "Bình thường", "Tiêu cực"] if c in topic_sentiment.columns]
topic_sentiment[ordered_cols].plot(
    kind="barh",
    stacked=True,
    ax=axes[1],
    color=[color_map[c] for c in ordered_cols],
    edgecolor="black"
)
axes[1].set_title("Phân bố Cảm xúc theo Chủ đề thảo luận", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Số lượng bài viết & bình luận")
axes[1].set_ylabel("Chủ đề")
axes[1].grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 💾 Cell 4: Lưu tập dữ liệu đã gán nhãn ra `data/cmc_dataset_labeled.csv`

In [ ]:
cols = ["id", "type", "time", "topic", "content", "cleaned_content", "likes", "label", "sentiment", "confidence"]
df[cols].to_csv(LABELED_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu {len(df):,} dòng dữ liệu có nhãn vào: {LABELED_CSV_PATH}")